In [1]:
import pandas as pd
import ir_datasets

qrels = (
    pd.DataFrame(ir_datasets.load("msmarco-passage/trec-dl-2020/judged").qrels_iter())
    .astype({"query_id": str, "doc_id": str})
    .drop("iteration", axis="columns")
    .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
    .merge(
        pd.read_parquet("../data/processed/trecdl-shallow-qrels-192.parquet")
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .rename({"relevance": "relevance_adapter_192"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet")
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .rename({"relevance": "relevance_adapter_256"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_parquet("../data/processed/trecdl-shallow-qrels-256.parquet")
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=0)
        .rename({"relevance": "relevance_baseline"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_json("../data/external/dl20-llm-qrels/gpt-4o_binary_dl20_few-shot.jsonl", lines=True)
        .query("prompt == 'few_shot_8'")
        .drop("prompt", axis=1)
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .rename({"relevance": "relevance_few_shot_8"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_csv("../data/external/dl20-llm-qrels/gpt-4o_binary_dl20-human.txt", header=None, names=["query_id", "q0", "doc_id", "relevance"], sep=" ")
        .drop("q0", axis="columns")
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .rename({"relevance": "relevance_gpt4_human"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
    .merge(
        pd.read_csv("../data/external/dl20-llm-qrels/gpt-4o_binary_dl20-umbrela.txt", header=None, names=["query_id", "q0", "doc_id", "relevance"], sep=" ")
        .drop("q0", axis="columns")
        .astype({"query_id": str, "doc_id": str})
        .assign(relevance=lambda df: (df["relevance"] > 0.5).astype(int))
        .rename({"relevance": "relevance_gpt4_umbrela"}, axis="columns"),
        how = "left",
        on = ["query_id", "doc_id"],
    )
)
qrels

,query_id,doc_id,relevance,relevance_adapter_192,relevance_adapter_256,relevance_baseline,relevance_few_shot_8,relevance_gpt4_human,relevance_gpt4_umbrela
0,23849,1020327,1,0.0,0.0,0.0,NaN,NaN,NaN
1,23849,1034183,1,NaN,NaN,NaN,NaN,0.0,0.0
2,23849,1120730,0,NaN,NaN,NaN,NaN,NaN,NaN
3,23849,1139571,1,0.0,0.0,0.0,0.0,NaN,NaN
4,23849,1139571,1,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
33363,1136962,937258,1,0.0,1.0,0.0,1.0,NaN,NaN
33364,1136962,937258,1,0.0,1.0,0.0,1.0,NaN,NaN
33365,1136962,937258,1,0.0,0.0,0.0,1.0,NaN,NaN
33366,1136962,937258,1,0.0,0.0,0.0,1.0,NaN,NaN


In [3]:
import krippendorff

df_reliability = []

for c in [
    'relevance_adapter_192',
    'relevance_adapter_256',
    'relevance_baseline',
    'relevance_few_shot_8',
    'relevance_gpt4_human',
    'relevance_gpt4_umbrela'
]:
    data = (
        qrels
        .dropna()
        .loc[:, ["relevance", c]]
        .values
        .T
    )
    alpha = krippendorff.alpha(reliability_data=data, level_of_measurement="nominal", value_domain=(0, 1))
    df_reliability.append({
        "qrels": c,
        "alpha": alpha,
    })

df_reliability = pd.DataFrame(df_reliability)
df_reliability

,qrels,alpha
0,relevance_adapter_192,0.499527
1,relevance_adapter_256,0.678104
2,relevance_baseline,-0.683150
3,relevance_few_shot_8,-0.057960
4,relevance_gpt4_human,-0.063552
5,relevance_gpt4_umbrela,-0.174161


In [7]:
import trectools
from glob import glob
from tqdm.notebook import tqdm

runs = [trectools.TrecRun(f) for f in glob(f"../data/external/dl20-passages-runs/*.gz")]
depths = [5, 10, 20, 50]
#eval_data = []

for c in [
    #'relevance_adapter_192',
    #'relevance_adapter_256',
    #'relevance_baseline',
    #'relevance_few_shot_8',
    #'relevance_gpt4_human',
    #'relevance_gpt4_umbrela'
    'relevance'
]:
    jdg = trectools.TrecQrel()
    jdg.qrels_data = (
        qrels
        .dropna()
        .loc[:, ["query_id", "doc_id", c]]
        .rename({"query_id": "query", "doc_id": "docid",c: "rel"}, axis="columns")
    )
    for r in tqdm(runs, desc=c):
        eval = trectools.TrecEval(r, jdg)
        eval_data.append(
            pd.concat([
                pd.concat([eval.get_ndcg(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"}),
                pd.concat(
                    [eval.get_unjudged(depth=k, per_query=True) for k in depths],
                    axis=1).reset_index().melt(id_vars=["query"]).rename(columns={"variable": "metric"})
            ])
            .reset_index()
            .assign(
                system=r.get_runid(),
                source=c
            )
        )

df = pd.concat(eval_data)

relevance:   0%|          | 0/59 [00:00<?, ?it/s]

In [9]:
(
    df
    .drop(["query", "index"], axis="columns")
    .dropna(subset="value")
    .astype({"value": float, "system": str, "source": str})
    .reset_index(drop=True)
    .groupby(["metric", "system", "source"])
    .mean()
    .reset_index()
    .pivot(index=["system", "metric"], columns="source", values="value")
    .reset_index()
    .set_index("system")
    .groupby("metric")
    .corr(method="spearman")
    .reset_index()
    .query("source == 'relevance'")
    .drop(["source", "relevance"], axis=1)
    #.query("metric in ['NDCG@5', 'NDCG@10', 'NDCG@20', 'NDCG@50']")
    #.set_index("metric")
    #.transpose()
    #.loc[:, ['NDCG@5', 'NDCG@10', 'NDCG@20', 'NDCG@50']]
)

source,metric,relevance_adapter_192,relevance_adapter_256,relevance_baseline,relevance_few_shot_8,relevance_gpt4_human,relevance_gpt4_umbrela
0,NDCG@10,0.877265,0.942314,NaN,0.746566,0.616868,0.329067
7,NDCG@20,0.784220,0.912624,NaN,0.802443,0.452218,0.498685
14,NDCG@5,0.881557,0.919755,NaN,0.758114,0.625528,0.382458
21,NDCG@50,0.909819,0.954763,NaN,0.729207,0.503828,0.451809
28,UNJ@10,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
35,UNJ@20,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
42,UNJ@5,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
49,UNJ@50,1.000000,1.000000,1.0,1.000000,1.000000,1.000000


In [167]:
print(few_shot_oss_8.query("query_id in @topics")["relevance"].value_counts())

relevance
0    1250
1     127
Name: count, dtype: int64


In [166]:
print(zero_qrels.query("query_id in @topics")["relevance"].value_counts())

relevance
0    3016
1     127
Name: count, dtype: int64


In [168]:
few_shot_oss_8

,query_id,doc_id,relevance
0,1131069,8772624,0
9,1131069,2723544,0
18,1131069,1780970,0
27,1131069,2996190,0
36,1131069,5538323,0
...,...,...,...
376,121171,2297409,0
381,121171,4355386,0
390,121171,5809828,0
391,121171,993579,0
